# 01 — Exploration des modèles de traduction FR → EN

**Objectif :** faire traduire le même jeu de 24 phrases par plusieurs modèles et relever,
pour chacun : les traductions, le temps d'inférence et la mémoire utilisée.

**Ce notebook ne choisit pas de modèle.** Il produit des données brutes que nous analyserons ensuite.
Les scores automatiques (BLEU, chrF) sont calculés dans `03_translation_evaluation.ipynb`.

**Avant de lancer :** *Exécution → Modifier le type d'exécution → GPU (T4)*.
Sans GPU, le notebook fonctionne quand même sur CPU, mais plus lentement.

## 1. Vérifier l'environnement

In [ ]:
import sys
import torch

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("GPU dispo   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))

## 2. Installer les dépendances

PyTorch est déjà installé sur Colab. On ajoute seulement :
- `transformers` : chargement des modèles Hugging Face ;
- `sentencepiece` : tokenizer utilisé par MarianMT, M2M100 et NLLB ;
- `sacremoses` : prétraitement recommandé pour MarianMT.

In [ ]:
!pip install -q transformers sentencepiece sacremoses

import transformers
print("transformers :", transformers.__version__)

## 3. Charger le jeu de test

Importe le fichier `notebooks/translation_test_set.json` du projet quand la fenêtre le demande.

In [ ]:
import json
from pathlib import Path

TEST_SET_PATH = Path("translation_test_set.json")

if not TEST_SET_PATH.exists():
    from google.colab import files
    print("Importe le fichier translation_test_set.json")
    files.upload()

if not TEST_SET_PATH.exists():
    raise FileNotFoundError(f"{TEST_SET_PATH} introuvable : vérifie le nom du fichier importé.")

with TEST_SET_PATH.open(encoding="utf-8") as f:
    test_set = json.load(f)

sentences = test_set["sentences"]
print(len(sentences), "phrases chargées (version", test_set["version"], ")")

## 4. Configuration de l'expérience

- `src_lang` : code de la langue source, utilisé par les modèles multilingues.
- `tgt_lang_token` : token qui force le décodeur à écrire en anglais.
  MarianMT n'en a pas besoin : il ne sait faire que FR → EN.

Les modèles plus gros (`m2m100_1.2B`, `nllb-200-distilled-1.3B`) sont en commentaire :
on les ajoutera si les premiers résultats le justifient.

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Identiques pour tous les modèles, pour une comparaison équitable.
GENERATION_PARAMS = {"num_beams": 4, "max_new_tokens": 256}

MODELS = [
    {"name": "Helsinki-NLP/opus-mt-fr-en",        "src_lang": None,       "tgt_lang_token": None},
    {"name": "Helsinki-NLP/opus-mt-tc-big-fr-en", "src_lang": None,       "tgt_lang_token": None},
    {"name": "facebook/m2m100_418M",              "src_lang": "fr",       "tgt_lang_token": "__en__"},
    {"name": "facebook/nllb-200-distilled-600M",  "src_lang": "fra_Latn", "tgt_lang_token": "eng_Latn"},
    # {"name": "facebook/m2m100_1.2B",             "src_lang": "fr",       "tgt_lang_token": "__en__"},
    # {"name": "facebook/nllb-200-distilled-1.3B", "src_lang": "fra_Latn", "tgt_lang_token": "eng_Latn"},
]

print("Appareil utilisé :", DEVICE)

## 5. Fonctions de chargement et de traduction

- `load_model` charge le tokenizer et le modèle **du même nom** (ils vont toujours ensemble).
- `translate_one` traduit **une phrase à la fois**, comme le fera notre pipeline segment par segment.

In [ ]:
import time
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


def load_model(config):
    """Charge tokenizer + modèle. Retourne aussi le temps de chargement (téléchargement inclus la 1re fois)."""
    start = time.perf_counter()

    tokenizer_kwargs = {}
    if config["src_lang"] is not None:
        tokenizer_kwargs["src_lang"] = config["src_lang"]
    tokenizer = AutoTokenizer.from_pretrained(config["name"], **tokenizer_kwargs)

    model = AutoModelForSeq2SeqLM.from_pretrained(config["name"]).to(DEVICE)
    model.eval()

    return tokenizer, model, time.perf_counter() - start


def get_target_lang_id(tokenizer, config):
    """Retourne l'id du token de langue cible, ou None pour MarianMT."""
    if config["tgt_lang_token"] is None:
        return None
    token_id = tokenizer.convert_tokens_to_ids(config["tgt_lang_token"])
    # Un code de langue inconnu donne silencieusement le token <unk> : on refuse ce cas.
    if token_id == tokenizer.unk_token_id:
        raise ValueError(f"Code de langue inconnu pour {config['name']} : {config['tgt_lang_token']}")
    return token_id


def translate_one(text, tokenizer, model, target_lang_id):
    """Traduit une phrase. Retourne (traduction, secondes d'inférence)."""
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)

    generation_kwargs = dict(GENERATION_PARAMS)
    if target_lang_id is not None:
        generation_kwargs["forced_bos_token_id"] = target_lang_id

    start = time.perf_counter()
    with torch.inference_mode():
        output_ids = model.generate(**inputs, **generation_kwargs)
    if DEVICE == "cuda":
        torch.cuda.synchronize()  # attendre la fin réelle du calcul GPU avant d'arrêter le chrono
    seconds = time.perf_counter() - start

    return tokenizer.decode(output_ids[0], skip_special_tokens=True), seconds


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


def model_size_mb(model):
    """Mémoire occupée par les poids du modèle (sans les calculs intermédiaires)."""
    return sum(p.numel() * p.element_size() for p in model.parameters()) / 1024**2

## 6. Lancer l'expérience

Pour chaque modèle : chargement → une traduction d'échauffement (non mesurée) → les 24 phrases → libération de la mémoire.

Le premier passage télécharge les modèles (environ 4 Go au total pour les 4 modèles actifs).

In [ ]:
import gc

results = []      # une ligne par (modèle, phrase)
model_stats = []  # une ligne par modèle

for config in MODELS:
    print("→", config["name"])
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()

    tokenizer, model, load_seconds = load_model(config)
    target_lang_id = get_target_lang_id(tokenizer, config)

    translate_one("Bonjour.", tokenizer, model, target_lang_id)  # échauffement

    times = []
    for sentence in sentences:
        translation, seconds = translate_one(sentence["text_fr"], tokenizer, model, target_lang_id)
        times.append(seconds)
        results.append({
            "model": config["name"],
            "id": sentence["id"],
            "category": sentence["category"],
            "text_fr": sentence["text_fr"],
            "reference_en": sentence["reference_en"],
            "translation_en": translation,
            "seconds": round(seconds, 3),
        })

    model_stats.append({
        "model": config["name"],
        "parameters_M": round(count_parameters(model) / 1e6, 1),
        "weights_MB": round(model_size_mb(model)),
        "gpu_peak_MB": round(torch.cuda.max_memory_allocated() / 1024**2) if DEVICE == "cuda" else None,
        "load_s": round(load_seconds, 1),
        "mean_s_per_sentence": round(sum(times) / len(times), 3),
        "max_s_per_sentence": round(max(times), 3),
        "total_s": round(sum(times), 1),
    })

    del model, tokenizer
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

print("Terminé.")

## 7. Statistiques par modèle

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", None)
stats_df = pd.DataFrame(model_stats)
stats_df

## 8. Traductions côte à côte

Une ligne par phrase, une colonne par modèle. C'est ici que se fait l'analyse humaine :
sens conservé ? anglais naturel ? noms propres intacts ? chiffres corrects ?

In [ ]:
results_df = pd.DataFrame(results)

comparison = results_df.pivot(index="id", columns="model", values="translation_en")
first_rows = results_df.drop_duplicates("id").set_index("id")
comparison.insert(0, "reference_en", first_rows["reference_en"])
comparison.insert(0, "text_fr", first_rows["text_fr"])
comparison.insert(0, "category", first_rows["category"])
comparison

## 9. Phrases coupées : morceaux traduits séparément vs phrase entière

Simule ce qui arrive quand Whisper coupe une phrase en deux segments.

In [ ]:
translation_of = {(r["model"], r["id"]): r["translation_en"] for r in results}
split_groups = sorted({s["split_group"] for s in sentences if "split_group" in s})

for group in split_groups:
    group_sentences = [s for s in sentences if s.get("split_group") == group]
    parts = sorted((s for s in group_sentences if s["part"] > 0), key=lambda s: s["part"])
    full = next(s for s in group_sentences if s["part"] == 0)

    print(f"=== Groupe {group} : {full['text_fr']}")
    for config in MODELS:
        name = config["name"]
        joined = " ".join(translation_of[(name, p["id"])] for p in parts)
        print(f"  {name}")
        print(f"    en morceaux : {joined}")
        print(f"    entière     : {translation_of[(name, full['id'])]}")
    print()

## 10. Sauvegarder les résultats

Trois fichiers sont téléchargés. Range-les dans `docs/experiments/` du projet :
ils servent de preuve de l'expérience et de base pour le notebook 03.

In [ ]:
from datetime import datetime, timezone

run_info = {
    "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
    "generation_params": GENERATION_PARAMS,
    "models": MODELS,
    "test_set_version": test_set["version"],
}

results_df.to_csv("01_translations.csv", index=False)
stats_df.to_csv("01_model_stats.csv", index=False)
with open("01_run_info.json", "w", encoding="utf-8") as f:
    json.dump(run_info, f, indent=2, ensure_ascii=False)

from google.colab import files
for filename in ["01_translations.csv", "01_model_stats.csv", "01_run_info.json"]:
    files.download(filename)